# Phase 3 - Measure the Cost of Restarting

**RHOAIENG-80952**

## What this notebook does

Every time a training job resizes (GPUs added or removed), PyTorch kills all
workers and restarts from the last checkpoint. This notebook measures how long
that restart takes and where the time goes.

The experiment:

1. Run a training job on 8 GPUs until it saves a checkpoint at step 100
2. Kill the job (simulating Kueue preemption)
3. Submit the same job again - it resumes from the checkpoint automatically
4. Measure the restart startup timeline and check loss after resume

## Why this is the most important measurement

The restart cost is the one number that decides whether elastic scaling can
ever pay for itself. If restarting takes 2 minutes, scaling up is worth it
whenever more than ~10 minutes of work remain. If it takes 20 minutes, elastic
scaling only helps very long jobs.

The research doc (H2) predicts that saving the checkpoint is cheap (LoRA
adapters are small) but restarting is expensive (container startup, model
loading, rendezvous). This experiment finds out exactly how expensive.

### Prerequisites

- IBM cluster workbench in `dhryshch-elastic-scaling` namespace
- PVC `elastic-scaling-shared` and Secret `hf-token`
- 8 GPUs available on the target node

Training code lives in `phase3/train.py`.

## Setup

In [ ]:
!pip install --force-reinstall --no-cache-dir -U \
    "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git@v0.3.0+rhaiv.2"
!pip install yamlmagic hatchling --index-url https://pypi.org/simple
!pip install --no-deps .. --index-url https://pypi.org/simple
%load_ext yamlmagic

## Configuration

Same model, dataset, and training settings as Phase 1. The key difference
is that `save_steps=100` creates checkpoints we can restart from.

In [ ]:
%%yaml parameters

# Infrastructure
namespace: dhryshch-elastic-scaling
pvc_name: elastic-scaling-shared
hf_secret_name: hf-token
target_node: oai-kft-ibm-jcsbk-gpu-2-8gmgw

# Model & Data
model_id: meta-llama/Llama-3.1-8B
dataset_id: tatsu-lab/alpaca
output_dir: /mnt/kubeflow-checkpoints

# Training
max_steps: 200
save_steps: 100
seq_length: 1024
seed: 42
global_batch_size: 128
per_device_batch_size: 4
lora_r: 16
lora_alpha: 32
warmup_steps_excluded: 20

# Phase 3 specific
gpus: 8

In [ ]:
%load_ext autoreload
%autoreload 2

from elastic_scaling_poc.phase3.train import train_func

print("train_func loaded")

In [ ]:
import os
from pathlib import Path

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubernetes import client as k8s

api_server = os.environ["OPENSHIFT_API_URL"]
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
if not token:
    sa_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
    if sa_path.exists():
        token = sa_path.read_text().strip()
if not token:
    raise RuntimeError(
        "Set NOTEBOOK_USER_TOKEN, or run inside a workbench "
        "with a service-account token."
    )

config = k8s.Configuration()
config.host = api_server
config.api_key = {"authorization": f"Bearer {token}"}
config.verify_ssl = False

client = TrainerClient(
    KubernetesBackendConfig(
        namespace=parameters["namespace"],
        client_configuration=config,
    )
)

In [ ]:
core_api = k8s.CoreV1Api(k8s.ApiClient(config))


def get_container_startup_time(job_name):
    """Time between pod creation and training container start."""
    pods = core_api.list_namespaced_pod(
        namespace=parameters["namespace"],
        label_selector=f"jobset.sigs.k8s.io/jobset-name={job_name}",
    )
    for pod in pods.items:
        created = pod.metadata.creation_timestamp
        for status in pod.status.container_statuses or []:
            if status.name != "node":
                continue
            state = status.state.terminated or status.state.running
            if state and state.started_at:
                delta = (state.started_at - created).total_seconds()
                print(f"  {pod.metadata.name}:")
                print(f"    created:   {created.isoformat()}")
                print(f"    started:   {state.started_at.isoformat()}")
                print(f"    container startup: {delta:.1f}s")

## Job submission helpers

In [ ]:
import time

from kubeflow.trainer.options import (
    ContainerOverride,
    Name,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow_trainer_api.models import IoK8sApimachineryPkgApiResourceQuantity

PACKAGES = ["datasets", "peft", "trl", "nvidia-ml-py"]


def pod_overrides():
    return PodTemplateOverrides(
        PodTemplateOverride(
            target_jobs=["node"],
            spec=PodSpecOverride(
                node_selector={"kubernetes.io/hostname": parameters["target_node"]},
                volumes=[
                    {
                        "name": "dshm",
                        "emptyDir": {
                            "medium": "Memory",
                            "sizeLimit": IoK8sApimachineryPkgApiResourceQuantity("16Gi"),
                        },
                    },
                    {
                        "name": "hf-token",
                        "secret": {"secretName": parameters["hf_secret_name"]},
                    },
                ],
                containers=[
                    ContainerOverride(
                        name="node",
                        volume_mounts=[
                            {"name": "dshm", "mountPath": "/dev/shm"},
                            {"name": "hf-token", "mountPath": "/mnt/hf-token", "readOnly": True},
                        ],
                    ),
                ],
            ),
        )
    )


def submit_job(name):
    trainer = TransformersTrainer(
        func=train_func,
        func_args=parameters,
        num_nodes=1,
        resources_per_node={"nvidia.com/gpu": parameters["gpus"]},
        packages_to_install=PACKAGES,
        output_dir=f"pvc://{parameters['pvc_name']}",
    )
    runtime = client.backend.get_runtime("torch-distributed")
    job_name = client.train(
        trainer=trainer,
        runtime=runtime,
        options=[pod_overrides(), Name(name)],
    )
    print(f"Submitted {job_name} ({parameters['gpus']} GPUs)")
    return job_name


def watch_job(name, poll_s=15, timeout_s=7200):
    seen, start = None, time.time()
    while time.time() - start < timeout_s:
        status = client.get_job(name).status
        if status != seen:
            print(f"  [{int(time.time() - start):5d}s] {status}")
            seen = status
        if status in ("Complete", "Failed"):
            return status
        time.sleep(poll_s)
    return "Timeout"


WORKBENCH_MOUNT = Path("/opt/app-root/src/elastic-scaling-shared")


def wait_for_checkpoint(step, poll_s=10, timeout_s=3600):
    """Wait until a checkpoint directory for the given step exists on the PVC."""
    run_name = f"phase3-restart-{parameters['gpus']}gpu"
    checkpoint_path = WORKBENCH_MOUNT / run_name / "checkpoints" / f"checkpoint-{step}"
    start = time.time()
    while time.time() - start < timeout_s:
        if checkpoint_path.exists():
            print(f"Checkpoint found: {checkpoint_path}")
            return True
        time.sleep(poll_s)
    print(f"Timeout waiting for {checkpoint_path}")
    return False

## Step 1 - Fresh training run

Start training on 8 GPUs. Wait for the checkpoint at step 100 to be saved,
then kill the job to simulate Kueue preemption. The checkpoint stays on the
PVC for the restart run to pick up.

The startup timeline from this run is the **fresh start** baseline - no
checkpoint to restore, everything loaded from scratch.

In [ ]:
fresh_job = submit_job("phase3-fresh")

In [ ]:
# Wait for checkpoint at step 100, then kill
wait_for_checkpoint(100)

In [ ]:
# Capture fresh run container startup before killing
print("Fresh run:")
get_container_startup_time("phase3-fresh")

## Step 2 - Kill the job

Simulate Kueue preemption. The job is killed mid-training (after step 100),
but the checkpoint remains on the PVC.

In [ ]:
client.delete_job(name=fresh_job)
print(f"Deleted {fresh_job}")

## Step 3 - Restart from checkpoint

Submit the same job with the same `max_steps=200`. The training script detects
checkpoint-100 on the PVC, resumes, and trains steps 101-200.

The startup timeline from this run shows the **restart cost** - how long it
takes to go from "container started" to "first useful training step".

In [ ]:
restart_job = submit_job("phase3-restart")

In [ ]:
restart_status = watch_job(restart_job)
print(f"Result: {restart_status}")

In [ ]:
for line in client.get_job_logs(restart_job, follow=False):
    if "[phase3]" in line:
        print(line, end="")

In [ ]:
# Capture restart run container startup
print("Restart run:")
get_container_startup_time("phase3-restart")

## Results

Read the restart startup timeline from the PVC.

In [ ]:
import json

import pandas as pd

phase3_dir = WORKBENCH_MOUNT / "phase3-restart-8gpu"

timing_file = phase3_dir / "startup_timing.json"
if timing_file.exists():
    timing = json.loads(timing_file.read_text())
    print(f"Run: {timing['run_name']}")
    print(f"Resumed from: {timing['resumed_from']}")
    print(f"Total training time: {timing['total_train_time_s']}s")
    print()
    print("Startup timeline:")
    df_timeline = pd.DataFrame(timing["timeline"])
    df_timeline
else:
    print(f"No timing file found at {timing_file}")

In [ ]:
# Check loss values after restart - does the loss look normal?
if timing_file.exists():
    losses = timing.get("losses", [])
    if losses:
        df_loss = pd.DataFrame(losses)[["step", "loss"]]
        print(f"Loss values after restart ({len(losses)} entries):")
        print(df_loss.to_string(index=False))
    else:
        print("No loss values recorded")

## Save & Cleanup

In [ ]:
import shutil

out = Path("../results/phase3")
out.mkdir(parents=True, exist_ok=True)

if timing_file.exists():
    shutil.copy(timing_file, out / "startup_timing.json")
    print(f"Saved to {out.resolve()}")

In [ ]:
for job_name in ["phase3-fresh", "phase3-restart"]:
    try:
        client.delete_job(name=job_name)
        print(f"Deleted {job_name}")
    except Exception as error:
        print(f"Skip {job_name}: {error}")